# Functions and pandas

## Packaging logic and working with tabular data

In the previous notebook we answered real questions about the constellation data, but each answer required its own loop, its own accumulator, its own block of code. If the battery classification thresholds changed, we would need to find and update every place we had written that logic.

The explainer article introduced two ideas that solve this. **Functions** let us write a piece of logic once, give it a name, and reuse it wherever we need it. That is the same principle behind `SUM` and `VLOOKUP` in a spreadsheet, except now we are the ones defining the function. **Libraries** are collections of functions other people have written and shared. The most important library for data analysis is **pandas**, which gives us a purpose-built toolkit for working with rows and columns.

By the end of this notebook we will be able to write our own functions, chain them together into a processing pipeline, and use pandas to load, inspect, and summarise the satellite telemetry dataset.

In [ ]:
%pip install -q pandas

In [ ]:
import pandas as pd

---

## 1. Why functions matter

In the previous notebooks we wrote battery-classification logic inline: "if battery is below 20, it's critical; if below 50, it's low," and so on. That works once. But the moment we need the same classification in three different places, we have three copies of the same code. Change the threshold in one place and forget the others, and we have a bug.

A **function** solves this. We write the logic once, give it a name, and call that name whenever we need it. The explainer compared this to a vending machine: we select an input, the machine does its work, and a result comes out. We do not need to know what happens inside. We just need to trust that it works.

## 2. Writing our first function

Here is the battery classifier from the previous notebooks, wrapped in a function. Instead of writing the if/elif/else block every time, we write it once and call `classify_battery` with any percentage value.

In [ ]:
def classify_battery(pct):
    """Classify a battery percentage into a health category."""
    if pct < 20:
        return "critical"
    elif pct < 50:
        return "low"
    elif pct < 80:
        return "good"
    else:
        return "excellent"

In [ ]:
print(classify_battery(12))   # critical
print(classify_battery(45))   # low
print(classify_battery(73))   # good
print(classify_battery(91))   # excellent

Four things to notice:

- **`def`** starts a function definition. The name follows, then parentheses with parameters.
- **`pct`** is a **parameter**: a placeholder for whatever value we pass in when we call the function.
- **`return`** sends a value back to the caller. Without it, the function returns `None`.
- The triple-quoted string on the second line is a **docstring**. It describes what the function does. Good practice is to write one for every function we create.

## 3. Functions with multiple parameters

Functions can take more than one input. We can also give parameters **default values**, so the caller does not have to specify every argument each time.

In [ ]:
def satellite_summary(sat_id, battery, signal, status="active"):
    """Return a one-line summary string for a satellite."""
    return f"{sat_id} | battery: {battery}% | signal: {signal} dBm | status: {status}"

In [ ]:
# All four arguments supplied
print(satellite_summary("SAT-003", 64, -72.5, "degraded"))

# status uses its default value
print(satellite_summary("SAT-011", 88, -55.1))

The `status="active"` in the function definition is a **default parameter value**. If the caller does not supply that argument, the function uses `"active"`. Parameters with defaults must come after parameters without defaults.

## 4. Functions that return values vs. print

This distinction trips people up early on. A function that **returns** a value gives us something we can store, compare, or pass into another function. A function that only **prints** produces text on the screen but gives back `None`. For analysis work, we almost always want `return`.

In [ ]:
# This function RETURNS a value
def battery_status_return(pct):
    """Return a status string."""
    if pct < 20:
        return "critical"
    return "ok"

# This function PRINTS a value
def battery_status_print(pct):
    """Print a status string."""
    if pct < 20:
        print("critical")
    else:
        print("ok")

In [ ]:
# The return version gives us a value we can use
result = battery_status_return(15)
print(f"Status is: {result}")
print(f"Is critical? {result == 'critical'}")

In [ ]:
# The print version gives us None
result = battery_status_print(15)
print(f"Status is: {result}")
print(f"Is critical? {result == 'critical'}")

The print version wrote "critical" to the screen, but `result` is `None`. We cannot compare it, store it, or pass it to another function. As a rule: **analysis functions should return values, not print them**. Print at the end, when presenting results to a human.

## 5. Combining functions into a pipeline

The explainer article talked about decomposition: breaking a problem into small, named steps. Here is what that looks like in practice. We will write three small functions (one for battery classification, one for signal classification, one that combines them into an overall health verdict) and then run a list of satellite readings through the whole pipeline.

In [ ]:
def classify_battery(pct):
    """Classify battery percentage into a health category."""
    if pct < 20:
        return "critical"
    elif pct < 50:
        return "low"
    elif pct < 80:
        return "good"
    else:
        return "excellent"


def classify_signal(dbm):
    """Classify signal strength in dBm."""
    if dbm > -60:
        return "strong"
    elif dbm >= -90:
        return "moderate"
    else:
        return "weak"


def overall_health(battery_pct, signal_dbm, status):
    """Determine overall satellite health from component readings."""
    battery_class = classify_battery(battery_pct)
    signal_class = classify_signal(signal_dbm)

    if status == "offline" or battery_class == "critical" or signal_class == "weak":
        return "red"
    elif status == "degraded" or battery_class == "low" or signal_class == "moderate":
        return "amber"
    else:
        return "green"

In [ ]:
# Test individual functions
print(classify_battery(15))     # critical
print(classify_signal(-95.0))   # weak
print(overall_health(85, -50, "active"))   # green
print(overall_health(15, -95, "offline"))  # red

In [ ]:
# Process a list of satellite readings through the pipeline
readings = [
    {"sat_id": "SAT-001", "battery": 85, "signal": -50.2, "status": "active"},
    {"sat_id": "SAT-004", "battery": 22, "signal": -88.1, "status": "active"},
    {"sat_id": "SAT-009", "battery": 11, "signal": -102.4, "status": "offline"},
    {"sat_id": "SAT-015", "battery": 67, "signal": -61.0, "status": "degraded"},
    {"sat_id": "SAT-018", "battery": 93, "signal": -45.8, "status": "active"},
]

for r in readings:
    health = overall_health(r["battery"], r["signal"], r["status"])
    bat = classify_battery(r["battery"])
    sig = classify_signal(r["signal"])
    print(f"{r['sat_id']}  battery={bat:<10} signal={sig:<10} overall={health}")

Each function is small and testable on its own. `overall_health` calls the other two, so the classification logic stays in one place. If the battery thresholds change, we update `classify_battery` once and everything that depends on it picks up the change automatically. This is exactly the advantage the explainer described: write it once, use it everywhere.

---

## 6. Introduction to pandas

Everything we have done so far (loading data, looping through rows, filtering, computing averages) works, but it takes many lines of code. **pandas** is a library that packages all of those operations into concise, expressive methods designed for tabular data.

We installed and imported it at the top of this notebook. The central object in pandas is the **DataFrame**: a two-dimensional table with labelled columns and an index. If we have worked with spreadsheet tables, a DataFrame is the programmatic equivalent, except it can handle millions of rows and every operation is recorded in our code.

## 7. Loading and inspecting data

One line loads 500 rows of CSV data into a DataFrame. Compare this to the manual approach with `open()`, `json.load()`, and a `for` loop from the previous notebook.

In [ ]:
df = pd.read_csv("../data/satellite_telemetry.csv")
df.head()

pandas gives us several ways to quickly understand what we are working with. Run each of the next few cells to see what they reveal about the dataset.

In [ ]:
# Last five rows
df.tail()

In [ ]:
# Dimensions: (rows, columns)
df.shape

In [ ]:
# Data types for each column
df.dtypes

In [ ]:
# Compact overview: types, non-null counts, memory usage
df.info()

In [ ]:
# Statistical summary of numeric columns
df.describe()

A few things to notice:

- `df.info()` shows the non-null count per column. Where the count is less than 500, there are missing values: gaps in `battery_pct`, `signal_strength_dbm`, and `temperature_c`.
- `df.describe()` gives us the mean, standard deviation, min, max, and quartiles for every numeric column in one call. This is the equivalent of writing a dozen spreadsheet formulas.
- pandas has automatically inferred the types: `float64` for decimal columns, `int64` for integers, `object` for strings.

## 8. Selecting columns and rows

Selecting data in pandas will feel familiar. Picking a column is like choosing a field in a spreadsheet. Filtering rows is like applying a filter, except we write the condition in code, so it is visible and reproducible.

In [ ]:
# Select a single column (returns a Series)
df["status"].head(10)

In [ ]:
# Select multiple columns (returns a DataFrame)
df[["satellite_id", "battery_pct"]].head()

In [ ]:
# Filter rows: only active satellites
active = df[df["status"] == "active"]
print(f"Active readings: {len(active)} of {len(df)}")
active.head()

In [ ]:
# Combine filters with & (and) — each condition must be in parentheses
active_low_battery = df[(df["status"] == "active") & (df["battery_pct"] < 30)]
print(f"Active satellites with low battery: {len(active_low_battery)}")
active_low_battery.head()

In [ ]:
# Combine filters with | (or)
needs_attention = df[(df["status"] == "offline") | (df["battery_pct"] < 20)]
print(f"Readings needing attention: {len(needs_attention)}")
needs_attention.head()

The syntax `df[df["column"] == value]` reads as: "give me the rows of `df` where this condition is true." The inner expression produces a series of `True`/`False` values, and pandas uses that to select rows. When combining conditions, wrap each one in parentheses and use `&` for AND, `|` for OR.

## 9. Basic aggregation

In a spreadsheet, we might use `AVERAGE`, `MIN`, `MAX`, and `COUNTIF` across a column. pandas provides the same operations as methods on a column (called a Series).

In [ ]:
# Summary statistics for a single column
print(f"Mean battery:   {df['battery_pct'].mean():.1f}%")
print(f"Median battery: {df['battery_pct'].median():.1f}%")
print(f"Min battery:    {df['battery_pct'].min():.1f}%")
print(f"Max battery:    {df['battery_pct'].max():.1f}%")
print(f"Std deviation:  {df['battery_pct'].std():.1f}%")

pandas skips `NaN` (missing) values automatically in these calculations. No need to filter them out first.

In [ ]:
# Count values in a categorical column
df["status"].value_counts()

In [ ]:
# Group by satellite and compute the mean battery for each
df.groupby("satellite_id")["battery_pct"].mean()

**`groupby`** is the pandas equivalent of a pivot table. It splits the DataFrame into groups (one per satellite), applies a function (here `.mean()`) to each group, and combines the results. This single line replaces what would have been a loop with a dictionary accumulator in the previous notebook.

## 10. Sorting and finding extremes

Sorting and finding the top or bottom values is straightforward. These operations answer the kinds of questions we are used to answering with spreadsheet sorts and conditional formatting.

In [ ]:
# Sort by battery percentage, lowest first
df.sort_values("battery_pct").head(10)

In [ ]:
# The 5 readings with the lowest battery
df.nsmallest(5, "battery_pct")

In [ ]:
# The 5 readings with the highest orbit altitude
df.nlargest(5, "orbit_altitude_km")

In [ ]:
# Which satellite has the lowest average battery?
avg_battery = df.groupby("satellite_id")["battery_pct"].mean()
worst_satellite = avg_battery.sort_values().head(1)
print(f"Lowest average battery:")
print(worst_satellite)

## 11. Applying functions to DataFrames

Here is where the two halves of this notebook come together. The `classify_battery` function we wrote earlier takes a single number and returns a string. The pandas `.apply()` method runs a function on every value in a column. That means we can create a new column by applying our function to an existing one, no loop required.

In [ ]:
# Apply our classify_battery function to every row
df["battery_class"] = df["battery_pct"].apply(classify_battery)
df[["satellite_id", "battery_pct", "battery_class"]].head(10)

In [ ]:
# How many readings fall into each battery class?
df["battery_class"].value_counts()

`classify_battery` was written without any knowledge of pandas. It takes a number and returns a string. `.apply()` is the bridge that connects plain Python functions to entire DataFrame columns. If `battery_pct` contains `NaN` (missing) values, the comparisons will behave unexpectedly. For production code we would add a check for that inside the function, but for now this is enough to see the pattern.

In [ ]:
# Apply classify_signal too
df["signal_class"] = df["signal_strength_dbm"].apply(classify_signal)
df[["satellite_id", "signal_strength_dbm", "signal_class"]].head(10)

---

## Exercises

These exercises bring together functions and pandas. If you get stuck on the pandas syntax, scroll back to the relevant section above.

### Exercise 1: Write a signal classifier

Write a function called `classify_signal` that takes a signal strength in dBm and returns:

- `"strong"` if the value is greater than -60
- `"moderate"` if the value is between -60 and -90 (inclusive of both ends)
- `"weak"` if the value is less than -90

Test it with values -45, -75, and -110.

In [ ]:
# Your code here


### Exercise 2: Filter degraded satellites

Load the satellite telemetry CSV into a DataFrame (or reuse `df` if it is still in memory). Filter to only the readings where `status` is `"degraded"`, then show the 5 rows with the lowest `battery_pct`.

In [ ]:
# Your code here


### Exercise 3: Average signal strength per satellite

Use `groupby` to calculate the mean `signal_strength_dbm` for each `satellite_id`. Sort the results from weakest (most negative) to strongest signal and display them.

In [ ]:
# Your code here


### Exercise 4: Maintenance flag

Write a function called `needs_maintenance` that takes a single row (a Series) and returns `True` if **any** of the following conditions are met:

- `battery_pct` is less than 30
- `signal_strength_dbm` is less than -100
- `status` is `"offline"`

Apply it to the DataFrame using `df.apply(needs_maintenance, axis=1)` to create a new column called `"maintenance"`. Then count how many readings are flagged for maintenance.

In [ ]:
# Your code here


---

## Summary

This notebook brought together two foundational skills.

**Functions** let us package logic for reuse:
- Define with `def`, pass data in through **parameters**, send results out with `return`.
- Add **docstrings** so our future selves (and our colleagues) know what a function does.
- Use **default parameter values** for optional arguments.
- Prefer `return` over `print`. Returned values feed into further computation.
- Compose small functions into a **pipeline** where each step calls the previous ones.

**pandas** gives us a concise, expressive toolkit for tabular data:
- `pd.read_csv()` loads a file into a **DataFrame** in one line.
- `.head()`, `.info()`, `.describe()` let us understand our data quickly.
- `df["column"]` selects a column; `df[condition]` filters rows.
- `.mean()`, `.median()`, `.min()`, `.max()`, `.value_counts()` for aggregation.
- `.groupby()` splits, applies, and combines: the programmatic equivalent of a pivot table.
- `.apply()` connects custom functions to DataFrame columns without writing a loop.

We started this unit storing a single variable and checking a single condition. We are now loading 500 rows of data, writing reusable functions, and applying them across an entire dataset. The analytical thinking we have built up over years of spreadsheet work is the same thinking that drives this code. The medium has changed, but the skills transfer directly.